## Introduction

The [NLBSE'27 tool competition](https://nlbse2027.github.io/tools/) on **code classification** consists of building and testing **a single multi-label classification model** that jointly identifies vulnerabilities and Self-Admitted Technical Debt (SATD). The model will predicts two binary labels: a vulnerability label for the code and an SATD label for the comments.

The competition provides a multi-annotated dataset based on MADE-WIC, divided into a public training set, a public validation set, and a hidden test set. We also provide a multitask BERT baseline with a shared encoder and a two-output classification head. Participants are invited to develop models that outperform the baseline while balancing classification performance, execution time, and computational cost.

The dataset and details about data preparation and baselines can be found in [our repository](https://github.com/nlbse2027/code-comment-classification) for the competition.

## Participation and submission acceptance

Please refer to our [tool competition page - comment classification](https://nlbse2027.github.io/tools/) to know how to participate and criteria for accepting submissions.


## Contact

Feel free to contact us with any questions you may have about the competition:

* Moritz Mock (momock@unibz.it)
* Thomas Borsani (tborsani@unibz.it)
* Barbara Russo (brusso@unibz.it)

## Training your classifiers

**The unit of classification for your classifiers is a code instance**.

Participants are free to select and transform the data from the provided **training dataset** under the following conditions:
1. No external information sources can be used to train/fine-tune the proposed classifiers. That means that **only the functions** can be used for classification. Participants can leverage this  code to improve classification.
2. While no new sources of information can be added/used, we allow using classification models pre-trained on other datasets but they must be fine-tuned on the provided training dataset.
3. Participants may preprocess, sample, apply over/under-sampling, select a subset of the attributes, perform feature engineering, split the training set into a model-fine-tuning validation set, etc.

Please contact us if you have any questions regarding these rules.

## Dataset Overview

We provide a dataset of 306,552 curated source-code functions derived from MADE-WIC. The functions originate from open-source projects represented in the OSPR, Big-Vul, and Devign datasets.
Each function has two binary labels:
1. a vulnerability label indicating whether the function is associated with a known code weakness or vulnerability; and
2. an MAT label representing the SATD-related annotation.

Because a function can be positive for either or both labels, the competition focuses on multi-label classification. Participants must build one classifier that predicts both labels jointly. I

Participants will use the dataset to train and validate their models and compare their results with the provided baselines. The organisers will evaluate the submitted models on a hidden test set.
Prediction labels

The labels are stored as a two-element binary vector in [Vulnerable, MAT] order:
Label	Meaning
[0, 0]	Neither vulnerable nor MAT-positive
[1, 0]	Vulnerable but not MAT-positive
[0, 1]	Not vulnerable but MAT-positive
[1, 1]	Both vulnerable and MAT-positive


The vulnerability label combines the original Devign, Big-Vul, and W annotations: a function is labeled vulnerable when at least one of these annotations is positive.
The competition requires participants to train and evaluate one multi-label classifier that predicts both labels. Separate classifiers for individual programming languages are not required.

## Training, validation, and test datasets
The dataset contains three stratified splits:
Split	Instances	Availability	Intended use
Training	214,586	Public	Model training
Test (public)	30,655	Public	Model tuning, selection, and validation reporting
Test	(hidden) 61,311	Hidden for the final evaluation by the organizsers, the delta between F1 (public) and F1 (hidden) is less than 0.5%.

Each row represents one source-code function and contains four columns:
- Projectname identifies the source project;
- Filepath contains the original source-file path;
- Function contains the formatted source-code function; and
- Label contains the two-element ground-truth vector in [Vulnerable, MAT] order.

Before splitting, functions are formatted consistently and functions exceeding the configured token limit are removed. The three splits preserve the joint distribution of the two labels using deterministic stratified sampling.
The public competition release will contain labels for the training and validation sets. Test labels will remain hidden and will be used only by the organisers to determine the final competition ranking.
Note: The current two-label CSV files contain Function but no separate developer-comment column. Therefore, the competition description should not claim that comments are provided as a distinct model input unless the dataset-generation pipeline is changed.

## Submission evaluation and ranking

The proposed classifiers will be evaluated based on their class-detection performance on the provided test sets for each of the language categories.

**Important:** you may apply any preprocessing or feature engineering on test dataset except sampling, rebalancing, undersampling, or oversampling techniques.

The classifiers should assign one single category label to the sentences in the **test set** for their corresponding category.
Classification performance must be measured by the $F_1$ score (defined below) achieved by each classifier (i.e., for each category in a language).

A submission (i.e., paper) in the tool competition must provide:
- Precision $P_c$ for each category $c$
- Recall $R_c$, for each category $c$
- $F_{1,c}$ score as the harmonic mean between $P_c$ and $R_c$, for each category $c$
- The average precision, recall, and $F_1$ scores across all 2 categories.

These metrics are defined as follows:
\begin{align}
P_c &= \frac{ TP_c }{ TP_c + FP_c } &
R_c &= \frac{ TP_c }{ TP_c + FN_c } &
F_{1,c} &= 2 \cdot \frac{ P_c \cdot R_c }{ P_c + R_c }
\end{align}

where $TP_c$, $FP_c$ and $FN_c$ represent the true positives, false positives, and false negatives for a category $c$, respectively.

The 19 scores of the proposed classifiers will be compared against the 19 $F_1$ scores achieved by the baseline classifier (see below) to rank the participants and determine a winner.

**To rank the competition submissions and determine a winner, we will use the following formula**:

\begin{align}
submission\_score(model) &= 0.60 \times avg. \space F_1
 + 0.2 \times max(\frac{(max\_avg\_runtime  - measured\_avg\_runtime)}{max\_avg\_runtime)}, 0)
 + 0.2 \times max(\frac{(max\_avg\_GFLOPS  - measured\_avg\_GFLOPS)}{max\_avg\_GFLOPS)}, 0)
 \end{align}


where:
- $avg. \space F_1$ is the average of the $F_1$ scores achieved by the proposed classifiers (implementing a single $model$, e.g., SVM, BERT, etc.) across the 2 categories.
- $max\_avg\_runtime$ indicates the maximum average inference runtime (using a Colab T4 instance -- read below) and the $measured\_avg\_runtime$ is the actual average inference runtime of the proposed classifiers. The runtime should be measured 10 times and should averaged across all categories and samples (i.e., code comments) -- **see the reference [notebook](https://colab.research.google.com/drive/1q1fGPpU7uxaK1mUVHa3b2xJOtEwhWwLG?usp=sharing) for the specific algorithm to measure the runtime**. Solutions with a runtime higher than $max\_avg\_runtime$ are allowed.
- $max\_avg\_GFLOPS$ indicates the maximum average compute in Giga FLOating Point operations per Second. Solutions with coumputational effort higher than $max\_avg\_GFLOPS$ are allowed.
- If a part of the submisson_score is negative, e.g., due to a high number of GFLOPS, is set to zero, such that no negative submussion-scores are possible

With this formula we encourage the participants to maximize the overall classification effectiveness and minimize the runtime of their models, for practical purposes.

**To account for hardware differences, the final evaluation must be run on a Google Colab T4 instance.** We provide a [base infrastructure (i.e, a notebook)](https://colab.research.google.com/drive/1q1fGPpU7uxaK1mUVHa3b2xJOtEwhWwLG?usp=sharing) to run classifiers on the test set, measure their runtime, and compute the submission score. This infrastructure can be reused by the competition participants.

The submission with the highest score, determined by our formula, will be the winner of the competition.

**Note**: we only allow training/testing a single ML/DL model (e.g., SVM or BERT) considering **all the categories**, rather than considering single categories. In other words, we do not allow using distinct models for subsets of categories.

